In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from gplearn.genetic import SymbolicTransformer
import warnings
warnings.filterwarnings('ignore')

# load actual training data rather than synthetic random matrix
train_path = 'playground-series-s6e3\\train.csv'
df = pd.read_csv(train_path)
# convert churn flag to numeric target and standardize column name
if 'Churn' in df.columns:
    df['target'] = df['Churn'].map({'Yes': 1, 'No': 0})
    df.drop(columns=['Churn'], inplace=True)
# ensure numeric columns are correct types
for col in df.select_dtypes(include='object').columns:
    if df[col].str.replace('.', '', 1).str.isnumeric().all():
        df[col] = pd.to_numeric(df[col])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
def create_bins(df, cols, n_bins=5):
    df_out = df.copy()
    for c in cols:
        # only bin numeric columns; leave others as NaN or skip
        if pd.api.types.is_numeric_dtype(df[c]):
            try:
                df_out[f'{c}_qcut'] = pd.qcut(df[c], q=n_bins, labels=False, duplicates='drop')
                df_out[f'{c}_cut'] = pd.cut(df[c], bins=n_bins, labels=False)
            except Exception:
                # fallback if qcut/cut fails (e.g., constant values)
                df_out[f'{c}_qcut'] = np.nan
                df_out[f'{c}_cut'] = np.nan
        else:
            df_out[f'{c}_qcut'] = np.nan
            df_out[f'{c}_cut'] = np.nan
    return df_out


def extract_digits(df, cols):
    df_out = df.copy()
    for c in cols:
        df_out[f'{c}_unit'] = (np.floor(df[c]) % 10).astype(int)
        df_out[f'{c}_tens'] = (np.floor(df[c] / 10) % 10).astype(int)
        df_out[f'{c}_dec'] = (np.floor(df[c] * 10) % 10).astype(int)
    return df_out

In [ ]:
def to_categorical(df, cols):
    df_out = df.copy()
    for c in cols:
        df_out[f'{c}_str'] = df[c].astype(str).astype('category')
    return df_out

def freq_encoding(df, cols):
    df_out = df.copy()
    for c in cols:
        freq = df[c].value_counts()
        df_out[f'{c}_freq'] = df[c].map(freq)
    return df_out

In [ ]:
class DVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, latent_dim * 2))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(), nn.Linear(64, input_dim))

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        x_noisy = x + 0.1 * torch.randn_like(x)
        h = self.encoder(x_noisy)
        mu, logvar = h.chunk(2, dim=-1)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def add_gp_features(X, y):
    gp = SymbolicTransformer(generations=3, population_size=100, random_state=42)
    gp.fit(X, y)
    return gp.transform(X)

In [ ]:
def train_cv_model(model_fn, X, y, fit_params=None):
    if fit_params is None: fit_params = {}
    oof = np.zeros(len(y))
    best_iters = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        model = model_fn()
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], **fit_params)
        
        oof[val_idx] = model.predict_proba(X_va)[:, 1]
        
        if hasattr(model, 'best_iteration'): best_iters.append(model.best_iteration)
        elif hasattr(model, 'best_iteration_'): best_iters.append(model.best_iteration_)
        elif hasattr(model, 'get_best_iteration'): best_iters.append(model.get_best_iteration())
            
    auc = roc_auc_score(y, oof)
    avg_iter = int(np.mean(best_iters)) if best_iters else None
    return oof, auc, avg_iter

In [6]:
# only use numeric columns for features (exclude target)
numeric_feats = df.select_dtypes(include=[np.number]).columns
features = [c for c in numeric_feats if c != 'target']

df_binned = create_bins(df, features)
df_digits = extract_digits(df_binned, features)

# datasets should contain only numeric columns as well
datasets = {
    'base': df[features],
    'binned': df_binned.select_dtypes(include=[np.number]).drop(columns=['target'], errors='ignore'),
    'digits': df_digits.select_dtypes(include=[np.number]).drop(columns=['target'], errors='ignore')
}
y = df['target']

oofs = {}
best_iterations = {}

for name, X_data in datasets.items():
    oofs[f'xgb_{name}'], _, best_iterations[f'xgb_{name}'] = train_cv_model(
        lambda: xgb.XGBClassifier(n_estimators=1000, eval_metric='auc', early_stopping_rounds=50), 
        X_data, y, {'verbose': False}
    )
    
    oofs[f'lgb_{name}'], _, best_iterations[f'lgb_{name}'] = train_cv_model(
        lambda: lgb.LGBMClassifier(n_estimators=1000), 
        X_data, y, {'callbacks': [lgb.early_stopping(50, verbose=False)]}
    )
    
    oofs[f'cb_{name}'], _, best_iterations[f'cb_{name}'] = train_cv_model(
        lambda: cb.CatBoostClassifier(iterations=1000, eval_metric='AUC', early_stopping_rounds=50), 
        X_data, y, {'verbose': False}
    )

df_oofs = pd.DataFrame(oofs)

In [7]:
def objective(trial):
    selected_cols = [c for c in df_oofs.columns if trial.suggest_categorical(f'use_{c}', [True, False])]
    if not selected_cols: return 0.0
        
    X_blend = df_oofs[selected_cols]
    blend_oof = np.zeros(len(y))
    
    for train_idx, val_idx in skf.split(X_blend, y):
        X_tr, y_tr = X_blend.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_blend.iloc[val_idx], y.iloc[val_idx]
        
        model = Ridge()
        model.fit(X_tr, y_tr)
        blend_oof[val_idx] = model.predict(X_va)
        
    return roc_auc_score(y, blend_oof)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=False)

best_cols = [c for c in df_oofs.columns if study.best_trial.params.get(f'use_{c}')]
print(f"Selected {len(best_cols)} models out of {len(df_oofs.columns)}")

[I 2026-03-04 23:02:54,397] A new study created in memory with name: no-name-cc9f8ba2-9043-4a5f-9e11-8b038800e600
[I 2026-03-04 23:02:54,843] Trial 0 finished with value: 0.9018418753273864 and parameters: {'use_xgb_base': False, 'use_lgb_base': False, 'use_cb_base': True, 'use_xgb_binned': True, 'use_lgb_binned': True, 'use_cb_binned': True, 'use_xgb_digits': False, 'use_lgb_digits': False, 'use_cb_digits': False}. Best is trial 0 with value: 0.9018418753273864.
[I 2026-03-04 23:02:55,289] Trial 1 finished with value: 0.9042583523173863 and parameters: {'use_xgb_base': True, 'use_lgb_base': False, 'use_cb_base': False, 'use_xgb_binned': True, 'use_lgb_binned': True, 'use_cb_binned': False, 'use_xgb_digits': True, 'use_lgb_digits': False, 'use_cb_digits': True}. Best is trial 1 with value: 0.9042583523173863.
[I 2026-03-04 23:02:55,761] Trial 2 finished with value: 0.9035942682288982 and parameters: {'use_xgb_base': True, 'use_lgb_base': True, 'use_cb_base': True, 'use_xgb_binned': Tru

Selected 5 models out of 9


In [8]:
X_final_blend = df_oofs[best_cols]
final_ridge = Ridge()
final_ridge.fit(X_final_blend, y)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


In [ ]:
full_models = []

for name in best_cols:
    model_type, feat_set = name.split('_')
    target_iter = int(best_iterations[name] * 1.25) if best_iterations[name] else 500
    X_train = datasets[feat_set]
    
    for seed in range(20):
        if model_type == 'xgb':
            m = xgb.XGBClassifier(n_estimators=target_iter, random_state=seed)
        elif model_type == 'lgb':
            m = lgb.LGBMClassifier(n_estimators=target_iter, random_state=seed)
        else:
            m = cb.CatBoostClassifier(iterations=target_iter, random_seed=seed, verbose=False)
            
        m.fit(X_train, y)
        full_models.append((m, feat_set))

[LightGBM] [Info] Number of positive: 133817, number of negative: 460377
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002677 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 840
[LightGBM] [Info] Number of data points in the train set: 594194, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225208 -> initscore=-1.235573
[LightGBM] [Info] Start training from score -1.235573
[LightGBM] [Info] Number of positive: 133817, number of negative: 460377
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002708 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 840
[LightGBM] [Info] Number of data points in the train set: 594194, number of used features: 5
[LightGBM] [Info

In [ ]:
test_df = pd.read_csv('playground-series-s6e3\\test.csv')
test_ids = test_df['id']
# restrict to numeric features for binning
numeric_cols = test_df.select_dtypes(include=[np.number]).columns
if 'id' in numeric_cols:
    numeric_cols = numeric_cols.drop('id')
test_features = numeric_cols.tolist()

test_binned = create_bins(test_df, test_features)
test_digits = extract_digits(test_binned, test_features)

test_datasets = {
    'base': test_df[test_features],
    # only numeric columns for bin/digits to avoid string features
    'binned': test_binned.select_dtypes(include=[np.number]).drop(columns=['id'], errors='ignore'),
    'digits': test_digits.select_dtypes(include=[np.number]).drop(columns=['id'], errors='ignore')
}

test_preds = {}
model_idx = 0

for name in best_cols:
    model_type, feat_set = name.split('_', 1) 
    X_test = test_datasets[feat_set]
    
    avg_preds = np.zeros(len(test_df))
    for seed in range(20):
        m, _ = full_models[model_idx]
        avg_preds += m.predict_proba(X_test)[:, 1]
        model_idx += 1
        
    test_preds[name] = avg_preds / 20.0

X_test_blend = pd.DataFrame(test_preds)[best_cols]

final_predictions = final_ridge.predict(X_test_blend)
final_predictions = np.clip(final_predictions, 0, 1) 

submission = pd.DataFrame({
    'id': test_ids,
    'Churn': final_predictions
})

submission.head()

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")

In [ ]:
# debugging: look at first few trained models and their feature names
print("best_cols", best_cols)
print("number of full models", len(full_models))
for i, (m, feat_set) in enumerate(full_models[:3]):
    print(i, type(m), feat_set)
    if hasattr(m, 'feature_names_'):
        print("feature_names_", m.feature_names_)
    try:
        print("get_feature_names", m.get_feature_names())
    except Exception as e:
        print("get_feature_names error", e)
    if hasattr(m, 'feature_names'):
        print("feature_names attr", m.feature_names)
